# FMATCH 6/10 — Product definitions (metadata review)

**PR topic**: the five FMATCH data-product definition YAMLs and the matching
`DataProductIdentifier`/`ProcessingStepIdentifier` additions in `constants.py`.
**This PR contains no algorithm code** — it is deliberately reviewable as science
metadata: variable names, dimensions, dtypes, units, long names.

| product | timescale / axis | source data |
|---|---|---|
| `FMATCH-CAM` | radiometer footprints (`RADIOMETER_TIME`) | camera + gridded ancillary |
| `FMATCH-CAM-CAMTIME` | camera pseudo-footprints (`CAMERA_TIME`, `FOOTPRINT`) | camera + gridded ancillary |
| `FMATCH-IMAGER` | radiometer footprints | RBSP CLDPIX/SSF + VIIRS + ancillary |
| `FMATCH-IMAGER-CAMTIME` | camera pseudo-footprints | RBSP + VIIRS + ancillary |
| `FMATCH-IMAGER-FLASH` | radiometer footprints | FLASHFlux (low-latency SSF) |

Conventions to review:

- Ancillary variables are named `<reader_key>_<field>` (e.g. `era5_wind_u10`,
  `igbp_surface_type`) with the instrument recorded in the `long_name`.
- Every continuous aggregated field carries a `_standard_deviation` companion.
- `cloud_fraction_camera` is in **percent** `[0, 100]`, matching the CF-CAM input.
- IMAGER-only extended SSF fields (layered cloud, assimilated aerosol, surface
  albedo, TOA incoming solar) appear only in the IMAGER-family products
  (`VariableSpec.only_modes` gating shown in PR 3).

In [1]:
import sys
from pathlib import Path

import pandas as pd

repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

In [2]:
from libera_utils.constants import DataProductIdentifier
from libera_utils.io.product_definition import LiberaDataProductDefinition

DEFS_DIR = repo_root / "libera_utils" / "data" / "product_definitions"
FMATCH_PRODUCTS = [
    "fmatch_cam",
    "fmatch_cam_camtime",
    "fmatch_imager",
    "fmatch_imager_camtime",
    "fmatch_imager_flash",
]

## 1. Every definition loads and validates against the Pydantic schema

In [3]:
definitions = {}
for name in FMATCH_PRODUCTS:
    definition = LiberaDataProductDefinition.from_yaml(DEFS_DIR / f"{name}.yml")
    definitions[name] = definition
    print(f"{name:24s} OK")

fmatch_cam               OK
fmatch_cam_camtime       OK
fmatch_imager            OK


fmatch_imager_camtime    OK
fmatch_imager_flash      OK


## 2. Product scope at a glance

In [4]:
rows = []
for name, definition in definitions.items():
    dims = {d for v in definition.variables.values() for d in v.dimensions}
    rows.append(
        {
            "product": name,
            "variables": len(definition.variables),
            "dimensions": ", ".join(sorted(dims)),
        }
    )
pd.DataFrame(rows).set_index("product")

,variables,dimensions
product,,
fmatch_cam,54,RADIOMETER_TIME
fmatch_cam_camtime,56,"CAMERA_TIME, FOOTPRINT"
fmatch_imager,518,RADIOMETER_TIME
fmatch_imager_camtime,100,"CAMERA_TIME, FOOTPRINT"
fmatch_imager_flash,66,RADIOMETER_TIME


## 3. Full variable inventory (example: FMATCH-CAM)

The same table can be rendered for any product by changing `which` — reviewers
should skim all five.

In [5]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", 90)


def inventory(which: str) -> pd.DataFrame:
    definition = definitions[which]
    return pd.DataFrame(
        [
            {
                "variable": var_name,
                "dtype": v.dtype,
                "dims": "x".join(v.dimensions),
                "units": v.attributes.get("units", ""),
                "long_name": v.attributes.get("long_name", ""),
            }
            for var_name, v in definition.variables.items()
        ]
    ).set_index("variable")


inventory("fmatch_cam")

,dtype,dims,units,long_name
variable,,,,
latitude,float32,RADIOMETER_TIME,degrees_north,Footprint boresight latitude
longitude,float32,RADIOMETER_TIME,degrees_east,Footprint boresight longitude
altitude,float32,RADIOMETER_TIME,meters,Spacecraft altitude above reference ellipsoid
solar_zenith_angle,float32,RADIOMETER_TIME,degrees,Solar zenith angle
viewing_zenith_angle,float32,RADIOMETER_TIME,degrees,Viewing zenith angle
relative_azimuth_angle,float32,RADIOMETER_TIME,degrees,Relative azimuth angle between Sun and sensor
sunglint_angle,float32,RADIOMETER_TIME,degrees,Sun glint angle
era5_wind_u10,float32,RADIOMETER_TIME,m/s,Eastward wind component at 10 m (ECMWF)
era5_wind_u10_standard_deviation,float32,RADIOMETER_TIME,m/s,Eastward wind component at 10 m (within-footprint standard deviation) (ECMWF)


## 4. What makes the IMAGER product different from CAM?

The diff below shows the variables FMATCH-IMAGER carries that FMATCH-CAM does not
(the RBSP CLDPIX/SSF cloud fields and the extended SSF cloud/aerosol/albedo set) and
vice versa (the camera cloud fraction).

In [6]:
cam_vars = set(definitions["fmatch_cam"].variables)
imager_vars = set(definitions["fmatch_imager"].variables)
print(f"IMAGER-only ({len(imager_vars - cam_vars)}):")
for v in sorted(imager_vars - cam_vars):
    print(f"  {v}")
print(f"\nCAM-only ({len(cam_vars - imager_vars)}):")
for v in sorted(cam_vars - imager_vars):
    print(f"  {v}")

IMAGER-only (465):
  cldpix_cloud_effective_height
  cldpix_cloud_effective_height_standard_deviation
  cldpix_cloud_effective_pressure
  cldpix_cloud_effective_pressure_standard_deviation
  cldpix_cloud_effective_temperature
  cldpix_cloud_effective_temperature_standard_deviation
  cldpix_cloud_mask
  cldpix_cloud_optical_depth
  cldpix_cloud_optical_depth_standard_deviation
  cldpix_cloud_particle_phase
  cldpix_cloud_particle_radius
  cldpix_cloud_particle_radius_standard_deviation
  cldpix_cloud_top_height
  cldpix_cloud_top_height_standard_deviation
  cldpix_cloud_water_path
  cldpix_cloud_water_path_standard_deviation
  era5_dew_point_temperature_2m
  era5_dew_point_temperature_2m_standard_deviation
  era5_forecast_albedo
  era5_forecast_albedo_standard_deviation
  era5_pressure_geopotential_1000hPa
  era5_pressure_geopotential_1000hPa_standard_deviation
  era5_pressure_geopotential_100hPa
  era5_pressure_geopotential_100hPa_standard_deviation
  era5_pressure_geopotential_10hPa
 

## 5. The units decision to double-check: camera cloud fraction in percent

In [7]:
for product in ("fmatch_cam", "fmatch_cam_camtime"):
    v = definitions[product].variables["cloud_fraction_camera"]
    print(f"{product:20s} units={v.attributes.get('units')!r}\n{'':22s}long_name={v.attributes.get('long_name')!r}")

fmatch_cam           units='percent'
                      long_name='Cloud fraction (Libera WFOV)'
fmatch_cam_camtime   units='percent'
                      long_name='Cloud fraction (Libera WFOV)'


## 6. New identifiers in `constants.py`

This PR adds the FLASH member of the FMATCH family (`FMATCH-IMAGER-FLASH`); the
scene-ID flash identifier arrives with the Scene-ID imager PR at the top of the
stack.

In [8]:
for dpi in DataProductIdentifier:
    if "FMATCH" in dpi.value:
        print(f"{dpi.name:28s} {dpi.value:24s} level={dpi.data_level}")

aux_fmatch_cam               FMATCH-CAM               level=AUX
aux_fmatch_cam_camtime       FMATCH-CAM-CAMTIME       level=AUX
aux_fmatch_imager            FMATCH-IMAGER            level=AUX
aux_fmatch_imager_camtime    FMATCH-IMAGER-CAMTIME    level=AUX
aux_fmatch_imager_flash      FMATCH-IMAGER-FLASH      level=AUX


## Cross-check with the readers (PRs 2–3)

The reader `product_variable_specs()` contracts and these YAMLs are locked together
by unit tests in the product-assembly PR (`test_product.py` cross-checks every
reader-contributed variable name/dtype against the YAML). Reviewers of *this* PR
should focus on the science content: names, units, long names, and dtypes.